> PostgreSQL 14+ practice. Run `sql/00_sample_data.sql` first, then `SET search_path TO sql_practice, public;` in this connection. Run cells in order in the same SQL session. See README.md for setup and expected results. These are SQL cells; the original Python notebook metadata did not make them executable as Python.


## 2 common method to create TEMP TABLE

## Define the schema manually

In [ ]:
-- Define a session-only table; dropping only pg_temp makes rerunning this example safe.
DROP TABLE IF EXISTS pg_temp.temp_transaction;
CREATE TEMP TABLE temp_transaction (
    transaction_id BIGINT, customer_id BIGINT, amount NUMERIC(18, 2),
    status TEXT, transaction_date DATE
);

In [ ]:
-- Insert one literal row into the manually defined temporary table, then inspect it.
INSERT INTO temp_transaction (transaction_id, customer_id, amount, status, transaction_date)
VALUES (101, 201, 15000, 'Success', DATE '2026-11-15');
SELECT * FROM temp_transaction;

## Create a temporary table from a query

In [ ]:
-- CREATE TABLE AS materializes successful raw payments as a separate session snapshot.
DROP TABLE IF EXISTS pg_temp.temp_successful_transaction;
CREATE TEMP TABLE temp_successful_transaction AS
SELECT transaction_id, customer_id, amount, status, transaction_date
FROM raw_transactions
WHERE status = 'Success';

### refence successful temp table above to Calculate total successful payment amount

In [ ]:
-- Aggregate the successful temporary snapshot by customer.
SELECT customer_id, SUM(amount) AS total_amount
FROM temp_successful_transaction
GROUP BY customer_id
ORDER BY customer_id;

### Temporary tables behave like regular tables in many respects. You can modify their records without modifying the original source dataset.

In [ ]:
-- Copy raw rows into a working table; edits to this snapshot do not edit raw_transactions.
DROP TABLE IF EXISTS pg_temp.temp_working_transaction;
CREATE TEMP TABLE temp_working_transaction AS
SELECT * FROM raw_transactions;

## insert new record to the above query

In [ ]:
-- Insert a new payment into the working copy only; rerun the notebook from its start before repeating.
INSERT INTO temp_working_transaction (transaction_id, customer_id, amount, status, transaction_date)
VALUES (999, 203, 35000, 'Success', DATE '2026-11-15');

### update the existing record from the above reference

In [ ]:
-- Update the working copy and compare with the unchanged raw source payment 105.
UPDATE temp_working_transaction
SET status = 'Success'
WHERE transaction_id = 105;
SELECT * FROM temp_working_transaction WHERE transaction_id IN (105, 999);
SELECT * FROM raw_transactions WHERE transaction_id = 105;

## Incremental extraction using a watermark

### Step 1 — Create an incremental temporary table

In [ ]:
-- Use an exclusive lower watermark and inclusive upper cutoff; missing watermark defaults to 1970.
DROP TABLE IF EXISTS pg_temp.temp_incremental_transaction;
CREATE TEMP TABLE temp_incremental_transaction AS
SELECT t.transaction_id, t.amount, t.updated_at
FROM raw_payment_event AS t
WHERE t.updated_at > (
    SELECT COALESCE(MAX(w.last_processed_at), TIMESTAMP '1970-01-01')
    FROM etl_watermarks AS w
    WHERE w.pipeline_name = 'payment_pipeline'
)
AND t.updated_at <= TIMESTAMP '2026-11-16 10:00:00';

### Step 2 — Validate the incremental batch

In [ ]:
-- Inspect batch size, time range and amount before loading; this example does not advance the watermark.
SELECT COUNT(*) AS total_transactions,
    MIN(updated_at) AS earliest_transaction,
    MAX(updated_at) AS latest_transaction,
    COALESCE(SUM(amount), 0) AS total_amount
FROM temp_incremental_transaction;